In [1]:
import sys
sys.path.append("../")

##### import library

In [2]:
from typing import List, Optional
import requests
import pandas as pd
import mygene
import utility
import importlib
importlib.reload(utility)
from utility import resolve_ensembl_ids_with_fallback, get_chembl_ids_fast,  get_disease_ids_fast, validate_id_dataframe
import pickle

In [3]:
dct_result = dict()

##### Read pickle

In [4]:
with open("Opentargets_same_question_response_final.pkl", "rb") as f:
    Opentarget_same_question_response = pickle.load(f)


In [5]:
# Opentarget_same_question_response

In [6]:
for model in ['llama-3.3-70b-versatile', 'gpt-5-nano', 'grok-4-1-fast-non-reasoning-latest', "gemini-2.5-flash-lite", "OpenTargets", "BioChatter", "ctd", "ttd", "hcdt"]:

    
    # if model=="llama-3.3-70b-versatile":
    #     dct_result[model] = Llama_same_question_response[model]

    # if model=="gpt-5-nano":
    #     dct_result[model] = OpenAI_same_question_response[model]

    # if model=='grok-4-1-fast-non-reasoning-latest':
    #     dct_result[model] = Grok_same_question_response[model]

    # if model=="gemini-2.5-flash-lite":
    #     dct_result[model] = Gemini_same_question_response[model]

    if model=="OpenTargets":
        dct_result[model] = Opentarget_same_question_response[model]

    # if model=="BioChatter":
    #     dct_result[model] = Biochatter_same_question_response[model]

    # if model=="ctd":
    #     dct_result[model] = CTD_same_question_response[model]

    # if model=="hcdt":
    #     dct_result[model] = HCDT_same_question_response[model]


In [7]:
# dct_result

##### Verify if only valid columns are present

In [8]:
# allowed = {"gene_name", "drug_name", "disease_name", "pathway_name"}

# missing_df = []
# not_dataframe = []
# empty_df = []
# bad_columns = []

# total_payloads = 0

# for model, queries in dct_result.items():
#     if not isinstance(queries, dict):
#         continue

#     for question, runs in queries.items():
#         if not isinstance(runs, dict):
#             continue

#         for run_id, payload in runs.items():
#             total_payloads += 1

#             if not isinstance(payload, dict) or "dataframe" not in payload:
#                 missing_df.append({
#                     "model": model,
#                     "question": question,
#                     "run_id": run_id,
#                     "payload_type": type(payload).__name__
#                 })
#                 continue

#             df = payload.get("dataframe")

#             if not isinstance(df, pd.DataFrame):
#                 not_dataframe.append({
#                     "model": model,
#                     "question": question,
#                     "run_id": run_id,
#                     "df_type": type(df).__name__
#                 })
#                 continue

#             if df.empty:
#                 empty_df.append({
#                     "model": model,
#                     "question": question,
#                     "run_id": run_id,
#                     "columns": list(df.columns)
#                 })

#             cols = {str(c) for c in df.columns}
#             extra = sorted(cols - allowed)

#             if extra:
#                 bad_columns.append({
#                     "model": model,
#                     "question": question,
#                     "run_id": run_id,
#                     "columns": list(df.columns),
#                     "extra_columns": extra
#                 })

# print(f"Total payloads checked: {total_payloads}")
# print(f"Missing dataframe key: {len(missing_df)}")
# print(f"Dataframe is not pd.DataFrame: {len(not_dataframe)}")
# print(f"Empty dataframes: {len(empty_df)}")
# print(f"Dataframes with disallowed columns: {len(bad_columns)}")

# bad_columns_df = pd.DataFrame(bad_columns)
# missing_df_df = pd.DataFrame(missing_df)
# not_dataframe_df = pd.DataFrame(not_dataframe)
# empty_df_df = pd.DataFrame(empty_df)

# display(bad_columns_df)
# display(missing_df_df)
# display(not_dataframe_df)
# display(empty_df_df)


In [9]:
dct_jaccard = dict()

question_entities = {
    "gene_name": set(),
    "drug_name": set(),
    "disease_name": set(),
}

allowed_cols = {"gene_name", "drug_name", "disease_name"}

for model, queries in dct_result.items():
    dct_jaccard[model] = {}
    print(f"\nWorking for model: {model}")

    for question, runs in queries.items():

        for run_id, payload in runs.items():
            df = payload.get("dataframe")
            # print(df.head())
            df.rename(columns={"gene_symbol": "gene_name"}, inplace=True)

            # ---- validation ----
            if not isinstance(df, pd.DataFrame) or df.empty:
                continue

            # ---- skip pathway outputs ----
            if "pathway_name" in df.columns:
                continue

            # ---- column sanity ----
            # if not set(df.columns).issubset(allowed_cols):
            #     continue

            # ---- collect entities globally ----
            for col in allowed_cols:
                if col in df.columns:
                    values = (
                        df[col]
                        .dropna()
                        .astype(str)
                        .str.strip()
                        .tolist()
                    )
                    question_entities[col].update(values)

# ---- finalize (sets → sorted lists) ----
for k in question_entities:
    question_entities[k] = sorted(question_entities[k])


Working for model: OpenTargets


In [10]:
question_entities

{'gene_name': ['7SK',
  'A1BG',
  'A1CF',
  'A2M',
  'A2M-AS1',
  'A2ML1',
  'A2MP1',
  'A3GALT2',
  'A4GALT',
  'A4GNT',
  'AAAS',
  'AACS',
  'AADAC',
  'AADACL2',
  'AADACL3',
  'AADACL4',
  'AADAT',
  'AAGAB',
  'AAK1',
  'AAMDC',
  'AAMP',
  'AANAT',
  'AAR2',
  'AARD',
  'AARS1',
  'AARS2',
  'AARSD1',
  'AASDH',
  'AASDHPPT',
  'AASS',
  'AATBC',
  'AATF',
  'AATK',
  'ABALON',
  'ABAT',
  'ABCA1',
  'ABCA10',
  'ABCA12',
  'ABCA13',
  'ABCA2',
  'ABCA3',
  'ABCA4',
  'ABCA5',
  'ABCA6',
  'ABCA7',
  'ABCA8',
  'ABCA9',
  'ABCB1',
  'ABCB10',
  'ABCB11',
  'ABCB4',
  'ABCB5',
  'ABCB6',
  'ABCB7',
  'ABCB8',
  'ABCB9',
  'ABCC1',
  'ABCC10',
  'ABCC11',
  'ABCC12',
  'ABCC13',
  'ABCC2',
  'ABCC3',
  'ABCC4',
  'ABCC5',
  'ABCC6',
  'ABCC6P1',
  'ABCC8',
  'ABCC9',
  'ABCD1',
  'ABCD2',
  'ABCD3',
  'ABCD4',
  'ABCE1',
  'ABCF1',
  'ABCF2',
  'ABCF3',
  'ABCG1',
  'ABCG2',
  'ABCG4',
  'ABCG5',
  'ABCG8',
  'ABHD1',
  'ABHD10',
  'ABHD11',
  'ABHD12',
  'ABHD12B',
  'ABHD13',
  

In [11]:
# question_entities

##### Associate id with all gene entry

In [12]:
# # Resolve unique gene symbols to IDs (MyGene -> OpenTargets fallback).
# df_gene = resolve_ensembl_ids_with_fallback(
#     list(question_entities["gene_name"]),
#     use_opentargets_fallback=True,
# )
# df_gene = df_gene.drop_duplicates(subset=["gene_name", "gene_id"]).reset_index(drop=True)
# df_gene


In [13]:
# df_gene[df_gene.isna().any(axis=1)]

##### Associate id with all drug entry

In [14]:
# # Resolve unique drug surface forms to IDs.
# # This keeps original raw names as rows and writes mapped ID + source.
# df_drug = await get_chembl_ids_fast(list(question_entities["drug_name"]))
# df_drug = df_drug.drop_duplicates(subset=["drug_name", "drug_id"]).reset_index(drop=True)
# df_drug


In [15]:
# df_drug["source"].value_counts()

In [16]:
# df_drug[df_drug["source"]=="Llama"]

In [17]:
# df_drug[df_drug.isna().any(axis=1)]

##### Associate id with all disease entry

In [18]:
# # Resolve unique disease surface forms to IDs.
# # Canonical fallback is internal; returned rows remain keyed by raw input disease_name.
# df_disease = await get_disease_ids_fast(list(question_entities["disease_name"]))
# df_disease = df_disease.drop_duplicates(subset=["disease_name", "disease_id"]).reset_index(drop=True)
# df_disease


In [19]:
# df_disease["source"].value_counts()

In [20]:
# df_disease[df_disease.isna().any(axis=1)]

In [21]:
# # ------------------------------------------------------------------
# # Disease canonical audit (OpenTargets-strict IDs only)
# # ------------------------------------------------------------------
# # IMPORTANT:
# # - This cell runs an additional Llama canonicalization pass over unresolved rows.
# # - IDs are still resolved ONLY via OpenTargets mapIds (score==1).
# # - To avoid changing the baseline resolver output, writeback is OFF by default.
# APPLY_AUDIT_MAPPINGS = False

# # 1) Rows unresolved after primary resolver
# unresolved_rows = df_disease[df_disease["disease_id"].isna()].reset_index(drop=True)
# unresolved_raw_names = list(
#     dict.fromkeys(
#         str(name).strip()
#         for name in unresolved_rows["disease_name"].dropna().astype(str).tolist()
#         if str(name).strip()
#     )
# )

# # 2) Llama canonicalization suggestions
# disease_input_map = {name: None for name in unresolved_raw_names}
# canonicalisation_map = await utility.canonicalise_disease_dict(disease_input_map)

# # 3) OpenTargets strict verification (score==1 only)
# canonical_terms_norm = list(
#     dict.fromkeys(
#         utility._normalize_disease_term(v)
#         for v in canonicalisation_map.values()
#         if isinstance(v, str) and v.strip()
#     )
# )
# canonical_to_id = utility.resolve_diseases_opentargets_bulk(
#     canonical_terms_norm,
#     strict_score_one=True,
# )

# # 4) Build persistent audit table
# audit_rows = []
# applied_count = 0
# candidate_count = 0

# for raw_disease_name in unresolved_raw_names:
#     canonical_name = canonicalisation_map.get(raw_disease_name)
#     if isinstance(canonical_name, str) and canonical_name.strip():
#         canonical_name = canonical_name.strip()
#         canonical_norm = utility._normalize_disease_term(canonical_name)
#         resolved_id = canonical_to_id.get(canonical_norm)
#     else:
#         canonical_name = None
#         canonical_norm = None
#         resolved_id = None

#     can_apply = bool(resolved_id)
#     if can_apply:
#         candidate_count += 1
#         print(f"[audit][disease][Llama->OT] raw='{raw_disease_name}' -> canonical='{canonical_name}' -> id='{resolved_id}'")

#     if can_apply and APPLY_AUDIT_MAPPINGS:
#         mask = df_disease["disease_name"].astype(str).str.strip() == raw_disease_name
#         df_disease.loc[mask, "disease_id"] = resolved_id
#         df_disease.loc[mask, "source"] = "Llama"
#         applied_count += 1

#     status = (
#         "mappable_ot_exact" if can_apply else
#         ("no_canonical" if canonical_name is None else "canonical_not_in_ot")
#     )
#     audit_rows.append({
#         "raw_disease_name": raw_disease_name,
#         "canonical_name": canonical_name,
#         "canonical_norm": canonical_norm,
#         "resolved_disease_id": resolved_id,
#         "status": status,
#         "can_apply": can_apply,
#         "applied": bool(can_apply and APPLY_AUDIT_MAPPINGS),
#     })

# disease_canonical_audit_df = pd.DataFrame(audit_rows)
# if not disease_canonical_audit_df.empty:
#     disease_canonical_audit_df = disease_canonical_audit_df.sort_values(
#         ["status", "raw_disease_name"],
#         ascending=[True, True],
#     ).reset_index(drop=True)

# print(f"[summary][disease][audit] OT-mappable candidates: {candidate_count}")
# print(f"[summary][disease][audit] applied to df_disease: {applied_count} (APPLY_AUDIT_MAPPINGS={APPLY_AUDIT_MAPPINGS})")
# if not disease_canonical_audit_df.empty:
#     print("[summary][disease][audit] status counts:")
#     print(disease_canonical_audit_df["status"].value_counts(dropna=False))

# disease_canonical_audit_df


In [22]:
# disease_canonical_audit_df

In [23]:
# df_disease[df_disease["source"]=='Llama']

In [24]:
# df_disease["source"].value_counts()

In [25]:
# df_disease

In [26]:
# df_disease[df_disease["source"]=="Llama"]

In [27]:
def compute_jaccard_from_run_dataframes(dct_result):
    """
    Compute per-question cross-run Jaccard consistency after mapping entities to IDs.

    For each (model, question):
    - intersection: entries present in ALL valid runs
    - union: entries present in ANY valid run
    - jaccard = |intersection| / |union|

    All comparisons are CASE-INDEPENDENT and ID-based.

    Side effects:
    - Stores per-run mapped dataframe in:
        dct_result[model][question][run]['dataframe_id']

    Returns:
    - dct_jaccard[model][question] = {
          'jaccard': float,
          'n_valid_runs': int,
          'intersection': set[tuple],
          'union': set[tuple],
      }
    """

    dct_jaccard = {}

    # # ------------------------------------------------------------------
    # # Build deterministic one-to-one mapping tables
    # # ------------------------------------------------------------------
    # gene_map = (
    #     df_gene.assign(_gene_norm=df_gene["gene_name"].astype(str).str.strip().str.upper())
    #     [["_gene_norm", "gene_id"]]
    #     .dropna(subset=["gene_id"])
    #     .drop_duplicates("_gene_norm", keep="first")
    # )

    # disease_map = (
    #     df_disease.assign(_disease_norm=df_disease["disease_name"].astype(str).str.strip().str.upper())
    #     [["_disease_norm", "disease_id"]]
    #     .dropna(subset=["disease_id"])
    #     .drop_duplicates("_disease_norm", keep="first")
    # )

    # drug_map = (
    #     df_drug.assign(_drug_norm=df_drug["drug_name"].astype(str).str.strip().str.upper())
    #     [["_drug_norm", "drug_id"]]
    #     .dropna(subset=["drug_id"])
    #     .drop_duplicates("_drug_norm", keep="first")
    # )

    # # ------------------------------------------------------------------
    # # Main loop
    # # ------------------------------------------------------------------
    for model, model_payload in dct_result.items():
        dct_jaccard[model] = {}
        print(f"\nWorking for model: {model}")

        for question_key, runs_dict in model_payload.items():
            run_sets = {}

            for run_number, run_payload in runs_dict.items():
                df = run_payload.get("dataframe")

                # ---- basic validity ----
                if not isinstance(df, pd.DataFrame) or df.empty:
                    continue

                if "pathway_name" in df.columns:
                    continue

    #             work_df = df.copy()

    #             # print(work_df.head())

    #             final_cols = []

    #             # ---- gene ----
    #             if "gene_name" in work_df.columns:
    #                 work_df["_gene_norm"] = (
    #                     work_df["gene_name"].astype(str).str.strip().str.upper()
    #                 )
    #                 work_df = work_df.merge(gene_map, on="_gene_norm", how="left")
    #                 final_cols.append("gene_id")

    #             # ---- disease ----
    #             if "disease_name" in work_df.columns:
    #                 work_df["_disease_norm"] = (
    #                     work_df["disease_name"].astype(str).str.strip().str.upper()
    #                 )
    #                 work_df = work_df.merge(disease_map, on="_disease_norm", how="left")
    #                 final_cols.append("disease_id")

    #             # ---- drug ----
    #             if "drug_name" in work_df.columns:
    #                 work_df["_drug_norm"] = (
    #                     work_df["drug_name"].astype(str).str.strip().str.upper()
    #                 )
    #                 work_df = work_df.merge(drug_map, on="_drug_norm", how="left")
    #                 final_cols.append("drug_id")

    #             if not final_cols:
    #                 continue

                # ---- canonical ID dataframe ----
                # id_df = (
                #     work_df[final_cols]
                #     .dropna(how="any")
                #     .drop_duplicates()
                #     .reset_index(drop=True)
                # )

                # id_df = df[[c for c in ["gene_id", "disease_id", "drug_id"]]]
                cols = ["gene_id", "disease_id", "drug_id"]

                id_df = df[[c for c in cols if c in df.columns]]

                dct_result[model][question_key][run_number]["dataframe_id"] = id_df

                if id_df.empty:
                    continue

                # ---- CASE-INDEPENDENT tuple set ----
                run_set = {
                    tuple(str(v).strip().upper() for v in row)
                    for row in id_df.itertuples(index=False, name=None)
                }

                if run_set:
                    run_sets[run_number] = run_set

            # ------------------------------------------------------------------
            # Compute intersection / union
            # ------------------------------------------------------------------
            n_valid_runs = len(run_sets)

            if n_valid_runs < 2:
                print(
                    f"Not enough valid runs for '{question_key}' "
                    f"({n_valid_runs}/{len(runs_dict)}). Skipping."
                )
                continue

            sets = list(run_sets.values())
            intersection = set.intersection(*sets)
            union = set.union(*sets)

            if not union:
                print(f"Empty union for '{question_key}', skipping.")
                continue

            jaccard = len(intersection) / len(union)

            dct_jaccard[model][question_key] = {
                "jaccard": jaccard,
                "n_valid_runs": n_valid_runs,
                "intersection": intersection,
                "union": union,
            }

            print(
                f"'{question_key}': "
                f"J={jaccard:.4f}, "
                f"|∩|={len(intersection)}, "
                f"|∪|={len(union)}, "
                f"runs={n_valid_runs}"
            )

    return dct_jaccard


dct_jaccard = compute_jaccard_from_run_dataframes(
    dct_result=dct_result,
    # df_gene=df_gene,
    # df_disease=df_disease,
    # df_drug=df_drug,
)


Working for model: OpenTargets
'Which diseases are associated with BRAF?': J=1.0000, |∩|=1558, |∪|=1558, runs=5
'Which genes are associated with amyotrophic lateral sclerosis?': J=1.0000, |∩|=1492, |∪|=1492, runs=5
'Which diseases are associated with CDK4?': J=1.0000, |∩|=1136, |∪|=1136, runs=5
Not enough valid runs for 'Which pathways are associated with the JAK2 gene?' (0/5). Skipping.
'Which drugs are used to treat rheumatoid arthritis?': J=1.0000, |∩|=251, |∪|=251, runs=5
'Which genes are associated with ovarian cancer?': J=1.0000, |∩|=8979, |∪|=8979, runs=5
'What are the known targets of regorafenib?': J=1.0000, |∩|=18, |∪|=18, runs=5
Not enough valid runs for 'Which pathways are associated with the STAT3 gene?' (0/5). Skipping.
'Which genes are associated with fever (pyrexia)?': J=1.0000, |∩|=951, |∪|=951, runs=5
'Which diseases are treated with bevacizumab?': J=1.0000, |∩|=280, |∪|=280, runs=5
'What are the known targets of cabozantinib?': J=1.0000, |∩|=2, |∪|=2, runs=5
Not eno

In [28]:
# Run Jaccard consistency computation across runs after ID grounding.
dct_jaccard = compute_jaccard_from_run_dataframes(
    dct_result=dct_result,
    # df_gene=df_gene,
    # df_disease=df_disease,
    # df_drug=df_drug,
)

# Flatten into a quick summary table for inspection.
rows = [
    {
        "model": model,
        "question": question,
        "jaccard": payload["jaccard"],
        "n_valid_runs": payload["n_valid_runs"],
        "intersection_size": len(payload["intersection"]),
        "union_size": len(payload["union"]),
    }
    for model, qmap in dct_jaccard.items()
    for question, payload in qmap.items()
]

jaccard_summary = pd.DataFrame(rows)
if jaccard_summary.empty:
    print("No valid Jaccard entries were produced. Check input runs and validation filters.")
else:
    display(jaccard_summary.sort_values(["model", "jaccard"], ascending=[True, False]).head(20))



Working for model: OpenTargets
'Which diseases are associated with BRAF?': J=1.0000, |∩|=1558, |∪|=1558, runs=5
'Which genes are associated with amyotrophic lateral sclerosis?': J=1.0000, |∩|=1492, |∪|=1492, runs=5
'Which diseases are associated with CDK4?': J=1.0000, |∩|=1136, |∪|=1136, runs=5
Not enough valid runs for 'Which pathways are associated with the JAK2 gene?' (0/5). Skipping.
'Which drugs are used to treat rheumatoid arthritis?': J=1.0000, |∩|=251, |∪|=251, runs=5
'Which genes are associated with ovarian cancer?': J=1.0000, |∩|=8979, |∪|=8979, runs=5
'What are the known targets of regorafenib?': J=1.0000, |∩|=18, |∪|=18, runs=5
Not enough valid runs for 'Which pathways are associated with the STAT3 gene?' (0/5). Skipping.
'Which genes are associated with fever (pyrexia)?': J=1.0000, |∩|=951, |∪|=951, runs=5
'Which diseases are treated with bevacizumab?': J=1.0000, |∩|=280, |∪|=280, runs=5
'What are the known targets of cabozantinib?': J=1.0000, |∩|=2, |∪|=2, runs=5
Not eno

,model,question,jaccard,n_valid_runs,intersection_size,union_size
0,OpenTargets,Which diseases are associated with BRAF?,1.0,5,1558,1558
1,OpenTargets,Which genes are associated with amyotrophic la...,1.0,5,1492,1492
2,OpenTargets,Which diseases are associated with CDK4?,1.0,5,1136,1136
3,OpenTargets,Which drugs are used to treat rheumatoid arthr...,1.0,5,251,251
4,OpenTargets,Which genes are associated with ovarian cancer?,1.0,5,8979,8979
5,OpenTargets,What are the known targets of regorafenib?,1.0,5,18,18
6,OpenTargets,Which genes are associated with fever (pyrexia)?,1.0,5,951,951
7,OpenTargets,Which diseases are treated with bevacizumab?,1.0,5,280,280
8,OpenTargets,What are the known targets of cabozantinib?,1.0,5,2,2
9,OpenTargets,Which diseases are associated with the NPM1 gene?,1.0,5,664,664


In [29]:
# Persist enriched run payloads with dataframe_id attached for each run.
with open("OpenTargets_same_question_response_with_id.pkl", "wb") as f:
    pickle.dump(dct_result, f)
